# conv-windowing-2d — worked example 2: Windowed conv2d with same-style zero padding

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-windowing-2d`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Padding extends the input with zeros before windowing so the output keeps a target spatial size. With pad `P` on every side and stride 1, `OH = (H + 2P) - KH + 1`. Pad first with `F.pad`, then the windowed-einsum recipe is identical — only the padded tensor's shape and strides change.

## Worked solution

**Step 1 — pad the input.** `F.pad(x, (P, P, P, P))` adds `P` zero columns left/right and `P` zero rows top/bottom. The last two pad args are the *width* axis, the first two are the *height* axis. After padding, `Hp = H + 2P`, `Wp = W + 2P`.

**Step 2 — recompute output shape on the padded tensor.** `OH = Hp - KH + 1`, `OW = Wp - KW + 1`. For `KH=KW=3, P=1` this gives same-size output.

**Step 3 — read strides of the *padded* tensor.** `F.pad` returns a fresh contiguous tensor, so call `.stride()` on `xp`, not `x` — using the original strides would be a classic bug.

**Step 4 — build the view and contract.** Same `as_strided` pattern (`stride=(s_b, s_ic, s_h, s_w, s_h, s_w)`) and same einsum as the unpadded case. The zeros at the border contribute nothing to the sums, which is exactly the convolution boundary behavior.

**Step 5 — verify against `F.conv2d(..., padding=P)`.** Matches because we reproduced the same zero-padded receptive fields.

In [ ]:
import torch.nn.functional as F
from einops import einsum

def conv2d_padded(x: Tensor, w: Tensor, P: int) -> Tensor:
    xp = F.pad(x, (P, P, P, P))
    B, IC, Hp, Wp = xp.shape
    OC, _, KH, KW = w.shape
    OH = Hp - KH + 1
    OW = Wp - KW + 1
    s_b, s_ic, s_h, s_w = xp.stride()
    windows = xp.as_strided(
        size=(B, IC, OH, OW, KH, KW),
        stride=(s_b, s_ic, s_h, s_w, s_h, s_w),
    )
    return einsum(windows, w, 'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow')

t.manual_seed(0)
x = t.randn(2, 3, 6, 6)
w = t.randn(5, 3, 3, 3)
out = conv2d_padded(x, w, P=1)
ref = F.conv2d(x, w, padding=1)
print(out.shape, bool(t.allclose(out, ref, atol=1e-4)))